In [21]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler

data = pd.read_csv('data/application_train_FE_raw.csv')


# Correlation between features and target
correlations = data.corr()['TARGET'].abs().sort_values(ascending=False)
print(correlations)

TARGET                               1.000000
EXT_SOURCE_2                         0.160470
DAYS_BIRTH                           0.078161
prev_refused_rate                    0.076024
prev_share_mean_all                  0.073508
                                       ...   
NAME_INCOME_TYPE_Businessman         0.001692
FLAG_EMAIL                           0.001648
NAME_FAMILY_STATUS_Separated         0.001202
CODE_GENDER_XNA                      0.001070
NAME_HOUSING_TYPE_Co.op.apartment    0.000465
Name: TARGET, Length: 75, dtype: float64


In [22]:
# New Features

def add_engineered_features(df, target="TARGET"):
    df = df.copy()
    eps = 1e-6

    # ---- Ratios / capacity ----
    df["credit_to_income"]  = df["AMT_CREDIT"]  / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_income"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_credit"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)
    df["income_per_member"] = df["AMT_INCOME_TOTAL"] / np.maximum(df["CNT_FAM_MEMBERS"], 1)
    df["income_per_child"]  = df["AMT_INCOME_TOTAL"] / (1 + np.maximum(df["CNT_CHILDREN"], 0))
    df["dsr_monthly"]       = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"]/12.0 + eps)

    # ---- Time conversions & contactability ----
    df["age_years"]    = -df["DAYS_BIRTH"] / 365.25
    df["reg_years"]    = -df["DAYS_REGISTRATION"] / 365.25
    df["id_years"]     = -df["DAYS_ID_PUBLISH"] / 365.25
    df["phone_years"]  = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df["contactability"] = df["FLAG_PHONE"] + df["FLAG_EMAIL"]

    # ---- Previous behavior & pricing ----
    df["prev_interest_per_credit"] = df["prev_interest_mean"] / (df["prev_amt_credit_mean"] + eps)
    df["prev_payments_ratio"]      = df["prev_last3_cnt_payment_mean"] / (df["prev_cnt_payment_mean"] + eps)
    df["recent_approval_momentum"] = df["prev_last3_approved_rate"] - df["prev_last5_approved_rate"]
    df["recent_credit_growth"]     = df["prev_last3_amt_credit_mean"] - df["prev_last5_amt_credit_mean"]
    df["prev_rate_spread"]         = df["prev_rate_mean_all"] - df["prev_rate_med_all"]

    # ---- Region transforms ----
    # If inputs may be standardized/negative, clip to keep log1p well-defined
    df["region_pop_log"]      = np.log1p(np.clip(df["REGION_POPULATION_RELATIVE"], 0, None))
    df["region_rating_x_pop"] = df["REGION_RATING"] * df["REGION_POPULATION_RELATIVE"]

    # ---- Magnitudes (log) & mild nonlinearity ----
    df["log_income"]  = np.log1p(np.clip(df["AMT_INCOME_TOTAL"], 0, None))
    df["log_credit"]  = np.log1p(np.clip(df["AMT_CREDIT"],       0, None))
    df["log_annuity"] = np.log1p(np.clip(df["AMT_ANNUITY"],      0, None))
    df["ext2_sq"]     = df["EXT_SOURCE_2"] ** 2

    # ---- Hygiene: replace inf -> NaN, then fill numerics (binary->0, continuous->median) ----
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols_no_t = [c for c in num_cols if c != target]

    bin_cols, cont_cols = [], []
    for c in num_cols_no_t:
        vals = pd.unique(df[c].dropna())
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            bin_cols.append(c)
        else:
            cont_cols.append(c)

    if bin_cols:
        df[bin_cols] = df[bin_cols].fillna(0)
    if cont_cols:
        df[cont_cols] = df[cont_cols].fillna(df[cont_cols].median())

    # Ensure target dtype if present; optional downcast for speed
    if target in df.columns:
        df[target] = df[target].astype(int)

    float_cols = [c for c in df.columns if c != target and pd.api.types.is_float_dtype(df[c])]
    df[float_cols] = df[float_cols].astype("float32")

    return df

data_fe = add_engineered_features(data)


df = data_fe.copy()
ID_COL = "SK_ID_CURR"


has_target = "TARGET" in df.columns
X = df.drop(columns=[ID_COL] + (["TARGET"] if has_target else []))

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

def is_dummy(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    vals = pd.unique(s)
    return len(vals) <= 2 and set(vals).issubset({0, 1, 0.0, 1.0})

dummy_cols = [c for c in num_cols if is_dummy(X[c])]
to_scale  = [c for c in num_cols if c not in dummy_cols]

scaler = StandardScaler()
X[to_scale] = scaler.fit_transform(X[to_scale])

data_fe = pd.concat([df[[ID_COL]], X] + ([df[["TARGET"]]] if has_target else []), axis=1)


In [23]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data_fe, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data_fe, test_size=0.2, random_state=42, stratify=data_fe['TARGET'])

# # For custom stratification, create bins for continuous variables first
# data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
# data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# # Create a combined stratification column
# data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# # Now stratify by the combined column
# train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

In [24]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    if P == 0 or N == 0:
        return 0.5
    order = np.argsort(-y_score, kind='mergesort')
    y_sorted = y_true[order]
    s_sorted = y_score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    changes = np.where(np.diff(s_sorted) != 0)[0]
    cut_idx = np.r_[changes, len(s_sorted) - 1]
    tpr = tp[cut_idx] / P
    fpr = fp[cut_idx] / N
    tpr = np.r_[0.0, tpr, 1.0]
    fpr = np.r_[0.0, fpr, 1.0]

    return float(np.trapz(tpr, fpr))

In [25]:
def choose_threshold_at_recall(y_true, y_score, target_recall=0.80):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(y_score, dtype=float)
    order = np.argsort(-s)
    y_sorted = y[order]
    s_sorted = s[order]
    P = int((y == 1).sum())
    N = int((y == 0).sum())
    if P == 0:
        return float(s_sorted.max() + 1e-12)
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    tpr = tp / max(P, 1)
    fpr = fp / max(N, 1)
    ok = np.where(tpr >= target_recall)[0]
    if len(ok) == 0:
        return float(s_sorted.min() - 1e-12)

    i = ok[np.argmin(fpr[ok])]
    return float(s_sorted[i] - 1e-12)


def cross_validate(model, train_data, val_data, cv=5, target_col='TARGET', target_recall=0.80):

    def get_scores_proba_or_decision(m, X_tr, X_te):
        if hasattr(m, "predict_proba"):
            s_tr = m.predict_proba(X_tr)[:, 1]
            s_te = m.predict_proba(X_te)[:, 1]
            return s_tr, s_te
        elif hasattr(m, "decision_function"):
            s_tr_raw = m.decision_function(X_tr).astype(float)
            s_te_raw = m.decision_function(X_te).astype(float)
            mn, mx = s_tr_raw.min(), s_tr_raw.max()
            rng = (mx - mn) if (mx > mn) else 1.0
            s_tr = (s_tr_raw - mn) / rng
            s_te = (s_te_raw - mn) / rng
            return s_tr, s_te
        else:
            s_tr = m.predict(X_tr).astype(float)
            s_te = m.predict(X_te).astype(float)
            return s_tr, s_te

    X = train_data.drop(columns=[target_col]).to_numpy()
    y = train_data[target_col].to_numpy().astype(int)
    val_X = val_data.drop(columns=[target_col]).to_numpy()
    val_y = val_data[target_col].to_numpy().astype(int)
    n = len(X)
    fold_size = n // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else n
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        model.fit(X_train_fold, y_train_fold)
        s_tr, s_va = get_scores_proba_or_decision(model, X_train_fold, X_val_fold)
        thr = choose_threshold_at_recall(y_train_fold, s_tr, target_recall=target_recall)
        y_pred = (s_va >= thr).astype(int)

        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, s_va))
    avg_metrics = {k: float(np.mean(v)) for k, v in metrics.items()}
    model.fit(X, y)
    s_tr_full, s_val = get_scores_proba_or_decision(model, X, val_X)
    thr_full = choose_threshold_at_recall(y, s_tr_full, target_recall=target_recall)
    y_val_pred = (s_val >= thr_full).astype(int)
    val_metrics = {
        'accuracy': accuracy(val_y, y_val_pred),
        'recall': recall(val_y, y_val_pred),
        'precision': precision(val_y, y_val_pred),
        'specificity': specificity(val_y, y_val_pred),
        'f1_score': f1_score(val_y, y_val_pred),
        'roc_auc': roc_auc(val_y, s_val),
        'threshold_used': thr_full
    }
    coefs = model.coef_ if hasattr(model, 'coef_') else None
    return avg_metrics, coefs, val_metrics

In [26]:
# Models
# Logistic Regression
log_reg = LogisticRegression(
    max_iter=3000, solver="lbfgs", penalty="l2", C=2.0,
    class_weight='balanced'
)

# Elastic-net LR
log_reg_en = LogisticRegression(
    max_iter=3000, solver="saga", penalty="elasticnet",
    l1_ratio=0.15, C=1.0, class_weight='balanced'
)

# Linear SVM
base_svm = LinearSVC(
    class_weight='balanced', C=0.5, tol=5e-4, max_iter=8000, dual="auto"
)
svm_model = CalibratedClassifierCV(estimator=base_svm, method="sigmoid", cv=3)

# LDA
lda_model = LDA(solver="lsqr", shrinkage="auto")


In [27]:
# Run Models
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

feature_sets = {
    'Model_1': predictors1,
    'Model_2': predictors2,
    'Model_3': predictors3,
    'Model_4': predictors4,
    'Model_5': predictors5,
    'Model_6': predictors6,
    'Model_7': predictors7,
    'Model_8': predictors8
}

# Model 1
train_df = train_data_random[feature_sets['Model_1'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_1'] + [target]].copy()
log_reg_metrics1 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics1

# Model 2
train_df = train_data_random[feature_sets['Model_2'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_2'] + [target]].copy()
log_reg_metrics2 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics2

# Model 3
train_df = train_data_random[feature_sets['Model_3'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_3'] + [target]].copy()
log_reg_metrics3 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics3

# Model 4
train_df = train_data_random[feature_sets['Model_4'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_4'] + [target]].copy()
log_reg_metrics4 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics4

# Model 5
train_df = train_data_random[feature_sets['Model_5'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_5'] + [target]].copy()
log_reg_metrics5 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics5

# Model 6
train_df = train_data_random[feature_sets['Model_6'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_6'] + [target]].copy()
log_reg_metrics6 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics6

# Model 7
train_df = train_data_random[feature_sets['Model_7'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_7'] + [target]].copy()
log_reg_metrics7 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics7

# Model 8
train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
log_reg_metrics8 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics8



/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.4541284376012843,
  'recall': 0.7999986514166441,
  'precision': 0.10895130094394538,
  'specificity': 0.42366438225244074,
  'f1_score': 0.19177674946416906,
  'roc_auc': 0.675532193598761},
 array([[-0.52100616, -0.03888576,  0.00180547,  0.31738692, -0.19461488]]),
 {'accuracy': np.float64(0.45214118107156825),
  'recall': np.float64(0.7932231067564809),
  'precision': np.float64(0.1064832575217844),
  'specificity': np.float64(0.42255042589739505),
  'f1_score': np.float64(0.18776121566448434),
  'roc_auc': 0.6770315545891211,
  'threshold_used': 0.40143735179299206})

In [28]:
print(log_reg_metrics1)
print(log_reg_metrics2)
print(log_reg_metrics3)
print(log_reg_metrics4)
print(log_reg_metrics5)
print(log_reg_metrics6)
print(log_reg_metrics7)
print(log_reg_metrics8)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

({'accuracy': 0.38725618220466795, 'recall': 0.7993955200586045, 'precision': 0.0978712572950809, 'specificity': 0.35095914717785714, 'f1_score': 0.17438543633255452, 'roc_auc': 0.6264745066711648}, array([[ 1.71844351e-04, -1.52001716e-01,  1.10347539e-01,
         3.01259031e-01,  7.28260921e-02, -1.11465764e-01,
        -2.75520058e-02,  2.01985016e-01,  2.66420127e-01,
         2.12055770e-01]]), {'accuracy': np.float64(0.38402750619215226), 'recall': np.float64(0.799755052051439), 'precision': np.float64(0.09617556090136972), 'specificity': np.float64(0.347960828064956), 'f1_score': np.float64(0.171702785020926), 'roc_auc': 0.6197487102004543, 'threshold_used': 0.4298737327851191})
({'accuracy': 0.36085352719050307, 'recall': 0.7993451617874874, 'precision': 0.09411659358838691, 'specificity': 0.3222246725654295, 'f1_score': 0.16840020482652227, 'roc_auc': 0.6168166653554493}, array([[-0.09551454, -0.05294937,  0.2319441 , -0.24135415,  0.05702797,
        -0.11690672,  0.06698948

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.4545032352077705, 'recall': 0.8001643684981875, 'precision': 0.1090364333368455, 'specificity': 0.42405920310639295, 'f1_score': 0.19191329294834905, 'roc_auc': 0.6755760499594127}, None, {'accuracy': np.float64(0.45236931299700167), 'recall': np.float64(0.7938354766278832), 'precision': np.float64(0.10658882859178863), 'specificity': np.float64(0.42274522304273143), 'f1_score': np.float64(0.18794249124078774), 'roc_auc': 0.6772073524248416, 'threshold_used': 0.0562152284189931})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


LDA Metrics: ({'accuracy': 0.4542995461544007, 'recall': 0.8005630866400019, 'precision': 0.10904114688170345, 'specificity': 0.42380117074357465, 'f1_score': 0.19193273719539725, 'roc_auc': 0.6755575209650553}, array([[-0.5935078 , -0.0386129 ,  0.0181529 ,  0.34852341, -0.21283554]]), {'accuracy': np.float64(0.4522226567592231), 'recall': np.float64(0.7932231067564809), 'precision': np.float64(0.10649784866672148), 'specificity': np.float64(0.4226389700543661), 'f1_score': np.float64(0.18778389871460324), 'roc_auc': 0.6769564248990422, 'threshold_used': 0.05254308238709635})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [29]:
predictors_AUC_7pp = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "prev_approved_rate","prev_app_count","DAYS_BIRTH","DAYS_REGISTRATION","REGION_RATING",
    "credit_to_income","annuity_to_credit","age_years","reg_years",
    "prev_rate_spread","prev_interest_per_credit","ext2_sq"
]

# 2) Recent-behavior heavy (recency + momentum + a few capacity anchors)
predictors_RECENT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_INCOME_TOTAL","debt_ratio","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","ext2_sq"
]

# 3) Affordability & capacity focus (clean signal for linear models)
predictors_CAPACITY = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "credit_to_income","annuity_to_income","annuity_to_credit","income_per_member","dsr_monthly",
    "REGION_RATING","age_years","prev_approved_rate","ext2_sq"
]

# 4) Stability + contactability + region interaction (often improves ranking)
predictors_STABILITY = [
    "EXT_SOURCE_2","REGION_RATING","REGION_POPULATION_RELATIVE","region_rating_x_pop",
    "DAYS_BIRTH","reg_years","id_years","phone_years","FLAG_PHONE","FLAG_EMAIL",
    "AMT_INCOME_TOTAL","debt_ratio","prev_approved_rate","AMT_CREDIT","ext2_sq"
]

# 5) Tiny, high-signal (9 vars) for speed + surprisingly good AUC
predictors_TINY9 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING"
]

# 6) Contract/property tilt + core finance (diversifies signal a bit)
predictors_CONTRACT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "NAME_CONTRACT_TYPE_Cash.loans","NAME_CONTRACT_TYPE_Revolving.loans","FLAG_OWN_REALTY_Y",
    "REGION_RATING","credit_to_income","annuity_to_credit","prev_approved_rate","ext2_sq"
]

# Model AUC 7pp
train_df = train_data_random[predictors_AUC_7pp + [target]].copy()
val_df   = val_data_random[predictors_AUC_7pp + [target]].copy()
log_reg_metrics9 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics9)

# Model RECENT
train_df = train_data_random[predictors_RECENT + [target]].copy()
val_df   = val_data_random[predictors_RECENT + [target]].copy()
log_reg_metrics10 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics10)

# Model Capacity
train_df = train_data_random[predictors_CAPACITY + [target]].copy()
val_df   = val_data_random[predictors_CAPACITY + [target]].copy()
log_reg_metrics11 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics11)

# Model Stability
train_df = train_data_random[predictors_STABILITY + [target]].copy()
val_df   = val_data_random[predictors_STABILITY + [target]].copy()
log_reg_metrics12 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics12)

# Model Tiny9
train_df = train_data_random[predictors_TINY9 + [target]].copy()
val_df   = val_data_random[predictors_TINY9 + [target]].copy()
log_reg_metrics13 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics13)

# Model Contract
train_df = train_data_random[predictors_CONTRACT + [target]].copy()
val_df   = val_data_random[predictors_CONTRACT + [target]].copy()
log_reg_metrics14 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics14)


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.47807452883758506, 'recall': 0.800597947944927, 'precision': 0.11358780611395154, 'specificity': 0.44967119515424886, 'f1_score': 0.1989419918359161, 'roc_auc': 0.6914946253122995}, array([[-0.39613577, -0.37663318,  0.27853036,  0.00488449,  0.29063889,
        -0.22420926, -0.03180531,  0.11474709,  0.01857805,  0.06541299,
         0.08164433, -0.16241329, -0.11474708, -0.01857803, -0.04958068,
        -0.03888904, -0.0922789 ]]), {'accuracy': np.float64(0.4768934949810976), 'recall': np.float64(0.7824045723617065), 'precision': np.float64(0.10992572198801227), 'specificity': np.float64(0.45038870884910304), 'f1_score': np.float64(0.1927680547173607), 'roc_auc': 0.686968694165919, 'threshold_used': 0.4063114836765633})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.4713771134931573, 'recall': 0.7993485369767782, 'precision': 0.11213529738782499, 'specificity': 0.4424890985166162, 'f1_score': 0.1966738312487857, 'roc_auc': 0.687335967435369}, array([[-3.86115587e-01, -2.21944326e-02, -1.04195253e-04,
         2.81221412e-01, -2.57025504e-01, -7.75073958e-02,
        -9.86890059e-02,  3.24745028e-02, -6.45881609e-03,
         7.77674711e-02, -3.07670736e-02, -5.56390691e-02,
        -1.27155464e-01]]), {'accuracy': np.float64(0.47112501629513753), 'recall': np.float64(0.790977750561339), 'precision': np.float64(0.10975160732999122), 'specificity': np.float64(0.4433760116169934), 'f1_score': np.float64(0.19275729990548673), 'roc_auc': 0.684764083826559, 'threshold_used': 0.40457940699091055})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.472908875955573, 'recall': 0.7995576996176302, 'precision': 0.11245800761646492, 'specificity': 0.44413863473049275, 'f1_score': 0.19717597772066914, 'roc_auc': 0.6880946528956178}, array([[-0.48463306,  0.28946158, -0.20899063,  0.03704119,  0.00747569,
        -0.24801684, -0.05134614, -0.03705158,  0.0738522 ]]), {'accuracy': np.float64(0.4733737452744101), 'recall': np.float64(0.785874668299653), 'precision': np.float64(0.10962726729120989), 'specificity': np.float64(0.4462625511342507), 'f1_score': np.float64(0.19241341396371636), 'roc_auc': 0.6846592442191031, 'threshold_used': 0.4048882074157228})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.4613065886452864, 'recall': 0.8000415902342098, 'precision': 0.11028710263602044, 'specificity': 0.4314686862713386, 'f1_score': 0.19384602296019132, 'roc_auc': 0.6825965557021647}, array([[-0.37671801, -0.43864424,  0.3158824 ,  0.31908809,  0.12286548,
        -0.35517335, -0.03735191,  0.06462853,  0.0260118 , -0.19556429,
        -0.18742304, -0.13417332]]), {'accuracy': np.float64(0.4599139616738365), 'recall': np.float64(0.7968973259848948), 'precision': np.float64(0.10828502482456384), 'specificity': np.float64(0.4306787795073403), 'f1_score': np.float64(0.19066223871849972), 'roc_auc': 0.682773945906934, 'threshold_used': 0.4040615366880434})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [30]:
predictors_CONTRACT_2 = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "NAME_CONTRACT_TYPE_Cash.loans","NAME_CONTRACT_TYPE_Revolving.loans","FLAG_OWN_REALTY_Y",
    "REGION_RATING","credit_to_income","annuity_to_credit","prev_approved_rate","ext2_sq"
]

train_df = train_data_random[predictors_CONTRACT_2 + [target]].copy()
val_df   = val_data_random[predictors_CONTRACT_2 + [target]].copy()
train_strat_df = train_data_stratified[predictors_CONTRACT_2 + [target]].copy()
val_strat_df   = val_data_stratified[predictors_CONTRACT_2 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

Log Reg: ({'accuracy': 0.4613065886452864, 'recall': 0.8000415902342098, 'precision': 0.11028710263602044, 'specificity': 0.4314686862713386, 'f1_score': 0.19384602296019132, 'roc_auc': 0.6825965557021647}, array([[-0.37671801, -0.43864424,  0.3158824 ,  0.31908809,  0.12286548,
        -0.35517335, -0.03735191,  0.06462853,  0.0260118 , -0.19556429,
        -0.18742304, -0.13417332]]), {'accuracy': np.float64(0.4599139616738365), 'recall': np.float64(0.7968973259848948), 'precision': np.float64(0.10828502482456384), 'specificity': np.float64(0.4306787795073403), 'f1_score': np.float64(0.19066223871849972), 'roc_auc': 0.682773945906934, 'threshold_used': 0.4040615366880434})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.46254502927192476, 'recall': 0.7991748114590662, 'precision': 0.11042785968986057, 'specificity': 0.4328910746013577, 'f1_score': 0.19403848458362832, 'roc_auc': 0.6825365603793463}, None, {'accuracy': np.float64(0.45976730543605787), 'recall': np.float64(0.7971014492753623), 'precision': np.float64(0.10827972493345164), 'specificity': np.float64(0.43050169119339815), 'f1_score': np.float64(0.19065986377950836), 'roc_auc': 0.6827967913477202, 'threshold_used': 0.056431929568276674})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.45854444274508654, 'recall': 0.7999385610361613, 'precision': 0.10948402597633464, 'specificity': 0.42856153422622667, 'f1_score': 0.19260253995361323, 'roc_auc': 0.6806546678318053}, None, {'accuracy': np.float64(0.45968582974840305), 'recall': np.float64(0.8084762865792129), 'precision': np.float64(0.1106171475907773), 'specificity': np.float64(0.42905004165706484), 'f1_score': np.float64(0.19460772407092541), 'roc_auc': 0.6900852572376747, 'threshold_used': 0.05616524080771297})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LDA Metrics: ({'accuracy': 0.4519651652891551, 'recall': 0.8001052167004212, 'precision': 0.10829739088829811, 'specificity': 0.4213862881626498, 'f1_score': 0.1907691348491148, 'roc_auc': 0.6782393672940211}, array([[-1.11475202, -0.35765289,  0.26968705,  0.35044482,  0.20255889,
        -0.20255889, -0.04672725,  0.06781044,  0.01905812, -0.15541579,
        -0.19875873,  0.56178888]]), {'accuracy': np.float64(0.4540151218876287), 'recall': np.float64(0.80343087790111), 'precision': np.float64(0.10902966066880289), 'specificity': np.float64(0.4233244110400085), 'f1_score': np.float64(0.19200347255715247), 'roc_auc': 0.6872099189623994, 'threshold_used': 0.05082189428266241})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [31]:
predictors_COEF_CORE20 = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL",
    "credit_to_income","annuity_to_credit","annuity_to_income",
    "age_years","reg_years","REGION_RATING",
    "prev_approved_rate","prev_app_count",
    "prev_rate_spread","prev_interest_per_credit","prev_cnt_payment_mean",
    "prev_last3_approved_rate","prev_last5_approved_rate",
    "ext2_sq","region_rating_x_pop"
]

predictors_COEF_RECENT_CAP = [
    "EXT_SOURCE_2","debt_ratio","credit_to_income","annuity_to_credit","dsr_monthly","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","recent_credit_growth",
    "prev_payments_ratio","REGION_RATING","ext2_sq"
]

predictors_COEF_TINY12 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING",
    "prev_last3_approved_rate","prev_app_count","ext2_sq"
]

predictors_COEF_NOISE_REDUCED = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "credit_to_income","annuity_to_credit","income_per_member",
    "age_years","REGION_RATING","prev_approved_rate","prev_app_count",
    "prev_last3_approved_rate","prev_rate_spread","prev_interest_per_credit",
    "region_rating_x_pop","ext2_sq"
]


feature_sets_new = {
    "COEF_CORE20": predictors_COEF_CORE20,
    "COEF_RECENT_CAP": predictors_COEF_RECENT_CAP,
    "COEF_TINY12": predictors_COEF_TINY12,
    "COEF_NOISE_REDUCED": predictors_COEF_NOISE_REDUCED,
}

for name, cols in feature_sets_new.items():
    train_df = train_data_random[cols + [target]].copy()
    val_df   = val_data_random[cols + [target]].copy()
    print(f"\n=== {name} ===")
    print("LR:",  cross_validate(log_reg, train_df, val_df, cv=5))
    print("SVM:", cross_validate(svm_model, train_df, val_df, cv=5))
    print("LDA:", cross_validate(lda_model, train_df, val_df, cv=5))



=== COEF_CORE20 ===


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LR: ({'accuracy': 0.4807551633694033, 'recall': 0.7992202232570558, 'precision': 0.11397088931390251, 'specificity': 0.45270682002416385, 'f1_score': 0.19948684087681792, 'roc_auc': 0.6938648017819068}, array([[-0.39619736,  0.28646096, -0.24364102,  0.14583712,  0.05674936,
        -0.12092999, -0.18618701,  0.2009362 , -0.24854485, -0.03656321,
         0.05907778, -0.20506987, -0.04812602, -0.05112094, -0.03812898,
         0.10244397,  0.03948098, -0.03832533, -0.09268845,  0.00815032]]), {'accuracy': np.float64(0.4801851127623517), 'recall': np.float64(0.7850581751377832), 'precision': np.float64(0.11085809817542444), 'specificity': np.float64(0.45373567798260994), 'f1_score': np.float64(0.1942816730652657), 'roc_auc': 0.690021582471078, 'threshold_used': 0.40546713111207383})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM: ({'accuracy': 0.4804618350613218, 'recall': 0.7994702769879791, 'precision': 0.11394061672163198, 'specificity': 0.45236566614328444, 'f1_score': 0.19944830876855235, 'roc_auc': 0.693783302763224}, None, {'accuracy': np.float64(0.47953330726111326), 'recall': np.float64(0.7846499285568483), 'precision': np.float64(0.11068559417201762), 'specificity': np.float64(0.45306274238962974), 'f1_score': np.float64(0.194004239426668), 'roc_auc': 0.690101516210335, 'threshold_used': 0.056992100722575996})


...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM: ({'accuracy': 0.47787492115281455, 'recall': 0.7991988382082909, 'precision': 0.11339402830932248, 'specificity': 0.44957853666780123, 'f1_score': 0.19860116795454535, 'roc_auc': 0.6914481521667966}, None, {'accuracy': np.float64(0.47575283535393037), 'recall': np.float64(0.7846499285568483), 'precision': np.float64(0.10995108835559624), 'specificity': np.float64(0.44895429350617155), 'f1_score': np.float64(0.19287506271951832), 'roc_auc': 0.6867658541241033, 'threshold_used': 0.057060153089022236})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LDA: ({'accuracy': 0.4769949573880258, 'recall': 0.7999842738671465, 'precision': 0.11330824215382065, 'specificity': 0.4485505169188434, 'f1_score': 0.19849354768483649, 'roc_auc': 0.6900264392622646}, array([[-0.88919175, -0.30388144,  0.2259614 ,  0.31845692,  0.0782599 ,
        -0.11054955,  0.03227843, -0.24168051,  0.07121088, -0.25432695,
        -0.03608997,  0.01496598, -0.04565957, -0.03282261,  0.01198029,
         0.34657975]]), {'accuracy': np.float64(0.4753291617781254), 'recall': np.float64(0.7819963257807716), 'precision': np.float64(0.10957924544492434), 'specificity': np.float64(0.4487240786980467), 'f1_score': np.float64(0.19222277972905166), 'roc_auc': 0.6851463943255887, 'threshold_used': 0.052527550048309835})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [32]:
# get all columns except ID
all_features = [col for col in data_fe.columns if col != 'SK_ID_CURR']
train_all = train_data_random[all_features].copy()
val_all = val_data_random[all_features].copy()

log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.540392237843993, 'recall': 0.7976797465542514, 'precision': 0.12717163140326665, 'specificity': 0.5177377094467154, 'f1_score': 0.21935896915660189, 'roc_auc': 0.7304287678278761}, array([[-3.74606392e-03,  3.94510646e-02, -2.74629988e-02,
        -1.10288025e-01, -1.47330647e-01, -4.76603923e-02,
         9.79934471e-02,  1.62770059e-02,  4.45710329e-02,
        -2.08492010e-02, -3.15505113e-02, -1.24824229e-02,
         3.90044215e-02, -4.04078184e-01,  4.33916090e-02,
         5.83594304e-02, -3.49317391e-02, -9.80017036e-02,
         2.86793509e-02, -5.19755076e-02, -1.46673061e-01,
         1.64546592e-01,  1.65856958e-01, -2.84685457e-02,
         5.62690769e-02,  5.39122506e-02,  9.80597235e-02,
        -7.01245057e-02, -1.46688349e-02,  1.37043774e-04,
        -2.40480636e-02, -2.15380126e-02, -3.39540805e-03,
        -1.32010740e-01, -1.61673265e-02,  2.09971048e-02,
        -6.03585856e-02,  9.11712715e-03,  2.39569730e-03,
         2.74391498e-01,  1.57637580

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.5404492769326043, 'recall': 0.7980288449240245, 'precision': 0.12722721190397496, 'specificity': 0.5177680047359666, 'f1_score': 0.21945571420341747, 'roc_auc': 0.7305296664114781}, None, {'accuracy': np.float64(0.5388476078738105), 'recall': np.float64(0.7917942437232088), 'precision': np.float64(0.12449051638370937), 'specificity': np.float64(0.5169030795657794), 'f1_score': np.float64(0.215153364024627), 'roc_auc': 0.7239673626707328, 'threshold_used': 0.05825329171597815})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.537039447558506, 'recall': 0.7978549376379293, 'precision': 0.12635525365628633, 'specificity': 0.514076835362175, 'f1_score': 0.21814936558039855, 'roc_auc': 0.728276270160298}, array([[-4.35506630e-03,  2.29013654e-02,  6.28099819e-03,
        -1.57608379e-01, -6.20284574e-02,  8.37855590e-03,
         9.30978761e-02,  1.77691450e-02,  4.38211692e-02,
        -2.75655824e-02, -4.49188785e-02,  1.92740732e-03,
         6.05725772e-02, -8.19339875e-01,  3.99107061e-02,
         8.42159135e-02, -2.58502515e-02, -5.02026140e-02,
         1.18175043e-02, -4.80386964e-02, -1.13634018e-01,
         2.05656539e-01,  1.27430897e-01, -2.87626879e-02,
         4.92441542e-02,  4.66628449e-02,  1.48956455e-01,
        -6.54977752e-02, -2.80184011e-02, -7.25953061e-03,
        -2.41647312e-02, -1.15198663e-02, -1.70547693e-02,
        -1.29406918e-01, -3.25839461e-02,  1.54454992e-02,
        -3.77145623e-02,  2.85033470e-03, -1.33799126e-02,
         2.99079027e-01,  1.72836637e-

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


Trying new feature engineering















In [33]:
def add_legal_features(df, target="TARGET"):
    """Engineered features that avoid protected/demographic/geographic attributes."""
    out = df.copy()
    eps = 1e-6

    # -------- capacity / affordability (no age) --------
    # (works on standardized data; still useful as transforms)
    out["credit_to_income"]  = out["AMT_CREDIT"]  / (np.abs(out["AMT_INCOME_TOTAL"]) + eps)
    out["annuity_to_income"] = out["AMT_ANNUITY"] / (np.abs(out["AMT_INCOME_TOTAL"]) + eps)
    out["annuity_to_credit"] = out["AMT_ANNUITY"] / (np.abs(out["AMT_CREDIT"]) + eps)
    out["dsr_monthly"]       = out["AMT_ANNUITY"] / (np.abs(out["AMT_INCOME_TOTAL"])/12.0 + eps)

    # softplus to make safe positive transforms on standardized variables
    softplus = lambda x: np.log1p(np.exp(np.clip(x, -20, 20)))
    out["sp_income"]  = softplus(out["AMT_INCOME_TOTAL"])
    out["sp_credit"]  = softplus(out["AMT_CREDIT"])
    out["sp_annuity"] = softplus(out["AMT_ANNUITY"])

    # -------- prior behavior / pricing (legal) --------
    out["prev_interest_per_credit"] = out["prev_interest_mean"] / (np.abs(out["prev_amt_credit_mean"]) + eps)
    out["prev_payments_ratio"]      = out["prev_last3_cnt_payment_mean"] / (np.abs(out["prev_cnt_payment_mean"]) + eps)
    out["recent_approval_momentum"] = out["prev_last3_approved_rate"] - out["prev_last5_approved_rate"]
    out["recent_credit_growth"]     = out["prev_last3_amt_credit_mean"] - out["prev_last5_amt_credit_mean"]
    out["prev_rate_spread"]         = out["prev_rate_mean_all"] - out["prev_rate_med_all"]

    # -------- nonlinear bumps (bounded / stable) --------
    out["ext2_sq"]           = out["EXT_SOURCE_2"] ** 2
    out["debt_ratio_sq"]     = out["debt_ratio"] ** 2
    out["cti_sq"]            = out["credit_to_income"] ** 2
    out["a2c_sq"]            = out["annuity_to_credit"] ** 2

    # -------- interactions anchored on model-strong signals --------
    out["ext2_x_dsr"]        = out["EXT_SOURCE_2"] * out["dsr_monthly"]
    out["ext2_x_cti"]        = out["EXT_SOURCE_2"] * out["credit_to_income"]
    out["ext2_x_approved"]   = out["EXT_SOURCE_2"] * out["prev_approved_rate"]
    out["pricing_x_capacity"]= out["prev_rate_spread"] * out["credit_to_income"]
    out["payhist_x_approved"]= out["prev_cnt_payment_mean"] * out["prev_approved_rate"]
    out["interest_x_burden"] = out["prev_interest_per_credit"] * out["annuity_to_credit"]
    out["recent_x_pay"]      = out["recent_approval_momentum"] * out["prev_last3_cnt_payment_mean"]
    out["approved_x_count"]  = out["prev_approved_rate"] * out["prev_app_count"]

    # hygiene
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    num_cols = [c for c in out.columns if c != target and pd.api.types.is_numeric_dtype(out[c])]
    out[num_cols] = out[num_cols].fillna(out[num_cols].median())

    return out

# Build the engineered (legal) frame
data_legal = add_legal_features(data)


train_data_random_2, val_data_random_2 = train_test_split(data_legal, test_size=0.2, random_state=42)
train_data_stratified_2, val_data_stratified_2 = train_test_split(data_legal, test_size=0.2, random_state=42, stratify=data_legal['TARGET'])


target = "TARGET"

predictors_LEGAL_CORE20 = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL",
    "credit_to_income","annuity_to_income","annuity_to_credit","dsr_monthly",
    "prev_approved_rate","prev_app_count","prev_cnt_payment_mean",
    "prev_rate_spread","prev_interest_per_credit","recent_approval_momentum",
    "recent_credit_growth","prev_payments_ratio","ext2_sq","debt_ratio_sq","pricing_x_capacity"
]

predictors_LEGAL_RECENT = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_INCOME_TOTAL",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread",
    "ext2_sq","ext2_x_dsr","recent_x_pay","approved_x_count"
]

predictors_LEGAL_CAPACITY = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "credit_to_income","annuity_to_income","annuity_to_credit","dsr_monthly",
    "sp_income","sp_credit","sp_annuity",
    "prev_approved_rate","prev_rate_spread","prev_interest_per_credit","ext2_sq","a2c_sq","cti_sq"
]

predictors_LEGAL_TINY12 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","prev_rate_spread",
    "prev_interest_per_credit","prev_cnt_payment_mean",
    "ext2_sq","pricing_x_capacity","approved_x_count","dsr_monthly"
]


train_df = train_data_random_2[predictors_LEGAL_CORE20 + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_CORE20 + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_CORE20 + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_CORE20 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

Log Reg: ({'accuracy': 0.30039351633980144, 'recall': 0.7991644401663205, 'precision': 0.08649366656746334, 'specificity': 0.2564562916440553, 'f1_score': 0.1560892695548389, 'roc_auc': 0.541406353700199}, array([[-9.15909530e-10,  8.47453820e-10, -5.14980112e-07,
         1.04826938e-05,  7.01140852e-09,  2.01937182e-09,
         1.57433790e-10,  2.85466798e-11,  1.88920548e-09,
        -2.43021163e-10,  5.13892593e-09,  1.48379819e-08,
        -2.69743975e-11, -9.72568609e-12, -2.50198560e-11,
        -5.32148825e-07,  4.68585200e-09, -8.29067368e-10,
         7.34841511e-10, -1.01168982e-10]]), {'accuracy': np.float64(0.2955612045365663), 'recall': np.float64(0.8026127781179833), 'precision': np.float64(0.08511743695205108), 'specificity': np.float64(0.2515716587862367), 'f1_score': np.float64(0.15391239675891494), 'roc_auc': 0.5394041072581733, 'threshold_used': 0.47526583645221493})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.30040986695661087, 'recall': 0.8011858929076574, 'precision': 0.08667346068078358, 'specificity': 0.2562953015205548, 'f1_score': 0.15642088617585456, 'roc_auc': 0.532112901426191}, None, {'accuracy': np.float64(0.2978425237909008), 'recall': np.float64(0.8058787507654623), 'precision': np.float64(0.08566407012823572), 'specificity': np.float64(0.25376755387911953), 'f1_score': np.float64(0.1548660416584945), 'roc_auc': 0.5341210340102427, 'threshold_used': 0.07622350788716745})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.2978188325967447, 'recall': 0.7999200214554776, 'precision': 0.08604046347438615, 'specificity': 0.25372065397956983, 'f1_score': 0.1553661863081509, 'roc_auc': 0.5313979142328203}, None, {'accuracy': np.float64(0.29917872506843957), 'recall': np.float64(0.81372351160444), 'precision': np.float64(0.08742979812216753), 'specificity': np.float64(0.25398401077765764), 'f1_score': np.float64(0.15789473684210525), 'roc_auc': 0.5444409967480175, 'threshold_used': 0.07664967997091136})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LDA Metrics: ({'accuracy': 0.30185195954533767, 'recall': 0.798328534228977, 'precision': 0.0863613855510443, 'specificity': 0.25825060788686227, 'f1_score': 0.15585851535775141, 'roc_auc': 0.5337729276048897}, array([[ 4.94530633e-12, -1.06110574e-11, -4.32661415e-07,
         6.25969367e-06,  9.82312071e-09, -2.57514560e-10,
         2.09022323e-11,  8.42650085e-12,  2.50826788e-10,
        -6.42523673e-12,  2.60918185e-11, -3.78856446e-10,
        -4.72900609e-14, -7.43447229e-12,  1.81023680e-13,
        -3.85545168e-07, -1.15060496e-08,  4.76828982e-12,
        -5.93661971e-12, -9.86380016e-13]]), {'accuracy': np.float64(0.3030243775257463), 'recall': np.float64(0.8117053481331988), 'precision': np.float64(0.0876997884913107), 'specificity': np.float64(0.2583447077801216), 'f1_score': np.float64(0.15829659949622166), 'roc_auc': 0.5424479185625445, 'threshold_used': 0.07573235131105682})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [34]:
train_df = train_data_random_2[predictors_LEGAL_RECENT + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_RECENT + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_RECENT + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_RECENT + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)



/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

Log Reg: ({'accuracy': 0.2994280362610075, 'recall': 0.7996205651552007, 'precision': 0.08642414477923127, 'specificity': 0.2553595366454078, 'f1_score': 0.15598556443066602, 'roc_auc': 0.51341043813074}, array([[-2.17721627e-12,  2.04382445e-12, -1.35358365e-07,
         1.98819861e-07, -8.38563797e-13, -7.68174721e-13,
        -7.03890757e-14,  3.88762849e-11,  4.14547453e-11,
        -9.40947832e-12, -6.43025194e-14, -1.97457835e-12,
        -3.97370588e-12, -3.09867278e-12, -2.47529234e-12]]), {'accuracy': np.float64(0.2962618954503976), 'recall': np.float64(0.7999591753419065), 'precision': np.float64(0.08496292763300525), 'specificity': np.float64(0.2525633533443128), 'f1_score': np.float64(0.1536109750122489), 'roc_auc': 0.5159309265350593, 'threshold_used': 0.481188737268222})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.3000391186061381, 'recall': 0.8000212165134994, 'precision': 0.08652981595718656, 'specificity': 0.25598845198058984, 'f1_score': 0.1561654621345909, 'roc_auc': 0.514045546742634}, None, {'accuracy': np.float64(0.297337374527441), 'recall': np.float64(0.8028169014084507), 'precision': np.float64(0.08533674709251866), 'specificity': np.float64(0.25348421257681203), 'f1_score': np.float64(0.15427461902053463), 'roc_auc': 0.5167877462376587, 'threshold_used': 0.0775979260850778})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.30016132989732097, 'recall': 0.7998797577743624, 'precision': 0.08630592969338564, 'specificity': 0.2562739419447165, 'f1_score': 0.1557977929765392, 'roc_auc': 0.5148594459720265}, None, {'accuracy': np.float64(0.2990483639681919), 'recall': np.float64(0.8062563067608476), 'precision': np.float64(0.08675164491541985), 'specificity': np.float64(0.2544980766844522), 'f1_score': np.float64(0.15664823746225934), 'roc_auc': 0.5225067763273822, 'threshold_used': 0.07739794373297561})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LDA Metrics: ({'accuracy': 0.3027930435184022, 'recall': 0.7985504374473725, 'precision': 0.08648949377136986, 'specificity': 0.25924965420481627, 'f1_score': 0.15607234786215315, 'roc_auc': 0.5174042693024522}, array([[-1.42093007e-14,  3.23625833e-15, -2.64485419e-07,
         3.30024090e-08, -8.56942278e-15, -8.91195164e-15,
         3.42528863e-16, -3.41529626e-13, -3.04406252e-13,
        -7.37619732e-12, -2.29049406e-17, -1.34856968e-14,
        -2.01542083e-13,  2.99388202e-14, -2.59202984e-14]]), {'accuracy': np.float64(0.300661582583757), 'recall': np.float64(0.8088799192734611), 'precision': np.float64(0.08717212579929531), 'specificity': np.float64(0.25602254799425667), 'f1_score': np.float64(0.15738322894783344), 'roc_auc': 0.5246063346106306, 'threshold_used': 0.07639317859043807})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [35]:

train_df = train_data_random_2[predictors_LEGAL_CAPACITY + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_CAPACITY + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_CAPACITY + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_CAPACITY + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)



/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

Log Reg: ({'accuracy': 0.29875175856237124, 'recall': 0.7998245098737752, 'precision': 0.08636323319077019, 'specificity': 0.2546130015289112, 'f1_score': 0.1558891250897027, 'roc_auc': 0.5382706814296354}, array([[-8.89741197e-10, -5.19417550e-07,  1.05146705e-05,
         6.94961950e-09,  8.27601756e-10,  1.95961563e-09,
         1.54877368e-10,  2.86915072e-11,  1.85852841e-09,
         9.10188866e-09,  9.10188866e-09,  9.10188866e-09,
        -2.41198418e-10, -2.62293666e-11, -9.70162925e-12,
        -8.06347908e-10,  1.02541440e-12,  7.80429564e-09]]), {'accuracy': np.float64(0.2929539825316126), 'recall': np.float64(0.807103490508267), 'precision': np.float64(0.08521735382228066), 'specificity': np.float64(0.24834865147248933), 'f1_score': np.float64(0.1541580568443214), 'roc_auc': 0.5365068535183701, 'threshold_used': 0.47625146322909795})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.30202310867482224, 'recall': 0.8017551159397952, 'precision': 0.086914056029025, 'specificity': 0.25799684464822226, 'f1_score': 0.15682403445710974, 'roc_auc': 0.5259502339656945}, None, {'accuracy': np.float64(0.2990483639681919), 'recall': np.float64(0.8038375178607879), 'precision': np.float64(0.08562172504511556), 'specificity': np.float64(0.25525509571623367), 'f1_score': np.float64(0.15475909769708404), 'roc_auc': 0.5321893724665034, 'threshold_used': 0.07661626802649488})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.3007316510816958, 'recall': 0.8001261288996722, 'precision': 0.08639450570364461, 'specificity': 0.25687245053462704, 'f1_score': 0.15594671254289816, 'roc_auc': 0.5273398737420684}, None, {'accuracy': np.float64(0.3023562768869769), 'recall': np.float64(0.8070635721493441), 'precision': np.float64(0.08720778087927425), 'specificity': np.float64(0.2580256323896974), 'f1_score': np.float64(0.1574068607191356), 'roc_auc': 0.5323919279685965, 'threshold_used': 0.07751088081522792})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mask
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:1037: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(self.var_), copy=False, constant_mask=constant_mas

LDA Metrics: ({'accuracy': 0.305616207812555, 'recall': 0.8001936032691779, 'precision': 0.08696744062060852, 'specificity': 0.2621817283457247, 'f1_score': 0.15688050544569007, 'roc_auc': 0.5334092515180032}, array([[-3.66013112e-07, -5.14897669e-07,  7.10587478e-06,
         3.35692919e-08, -4.40026492e-07,  5.32706682e-05,
         1.84668948e-06, -1.12417940e-09,  2.21604202e-05,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         4.18020201e-07, -1.87899229e-09, -7.39328597e-12,
        -3.45765704e-07,  1.13122940e-09,  1.15451249e-03]]), {'accuracy': np.float64(0.30683743970799116), 'recall': np.float64(0.8084762865792129), 'precision': np.float64(0.08786051102094528), 'specificity': np.float64(0.26277631042490207), 'f1_score': np.float64(0.1584965380811078), 'roc_auc': 0.5423376427590931, 'threshold_used': 0.07626101511951547})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [36]:
train_df = train_data_random_2[predictors_LEGAL_TINY12 + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_TINY12 + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_TINY12 + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_TINY12 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

Log Reg: ({'accuracy': 0.12294039781921055, 'recall': 0.9096800070811399, 'precision': 0.07807186308879246, 'specificity': 0.053635345640910694, 'f1_score': 0.1437959844726691, 'roc_auc': 0.4638336819647845}, array([[-2.18719393e-19,  1.40290877e-19, -1.14003096e-19,
        -1.64063496e-19,  2.03315632e-21, -5.43469396e-21,
        -1.00946270e-11,  1.51604159e-18, -1.83748657e-19,
        -2.10263421e-20, -4.73100674e-19,  1.12707519e-19]]), {'accuracy': np.float64(0.12063290314170251), 'recall': np.float64(0.9022249438660951), 'precision': np.float64(0.07633060477325321), 'specificity': np.float64(0.05282544404894721), 'f1_score': np.float64(0.1407531247512141), 'roc_auc': 0.46758950210208966, 'threshold_used': 0.4999999999971647})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.12114394259662356, 'recall': 0.9140669196715301, 'precision': 0.07822138775147791, 'specificity': 0.05126595850764555, 'f1_score': 0.14410353321702635, 'roc_auc': 0.463833719372451}, None, {'accuracy': np.float64(0.1288130621822448), 'recall': np.float64(0.8914064094713207), 'precision': np.float64(0.07621557471464972), 'specificity': np.float64(0.06265384547273725), 'f1_score': np.float64(0.1404247793301928), 'roc_auc': 0.46758744528946194, 'threshold_used': 0.08099623422029405})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

...

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

SVM Metrics: ({'accuracy': 0.12975218508389405, 'recall': 0.8940628466035901, 'precision': 0.07729548532925326, 'specificity': 0.06262458221530878, 'f1_score': 0.14228687929493197, 'roc_auc': 0.4646053541992726}, None, {'accuracy': np.float64(0.13239799243905617), 'recall': np.float64(0.8914228052472251), 'precision': np.float64(0.07732572388921957), 'specificity': np.float64(0.06572953042738376), 'f1_score': np.float64(0.14230713468756545), 'roc_auc': 0.4642516808295202, 'threshold_used': 0.08077187770912093})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

LDA Metrics: ({'accuracy': 0.08551829262170507, 'recall': 0.9878547842441179, 'precision': 0.0802959723800681, 'specificity': 0.006271869193328039, 'f1_score': 0.14851723137237607, 'roc_auc': 0.4646044280420127}, array([[-2.37273924e-24, -2.22951024e-25,  7.11356574e-23,
        -2.02917133e-23,  1.19413313e-25,  1.81978992e-26,
        -7.62914251e-12,  3.93132289e-24, -2.18902924e-24,
        -3.74680970e-26,  2.94500275e-22, -3.28499084e-24]]), {'accuracy': np.float64(0.08510950332420805), 'recall': np.float64(0.9882946518668012), 'precision': np.float64(0.08029975075429621), 'specificity': np.float64(0.005778809848793718), 'f1_score': np.float64(0.14853121824716783), 'roc_auc': 0.4642508061357994, 'threshold_used': 0.08076261457182214})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


In [37]:
# get all columns except ID
all_features = [col for col in data_legal.columns if col != 'SK_ID_CURR']
train_all = train_data_random_2[all_features].copy()
val_all = val_data_random_2[all_features].copy()

log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-lear

({'accuracy': 0.33541660044310184, 'recall': 0.7997712873250384, 'precision': 0.09079958660574058, 'specificity': 0.2945063530736317, 'f1_score': 0.16308004511720617, 'roc_auc': 0.5809214937188234}, array([[ 8.15602345e-07,  4.24557959e-10,  7.65787877e-08,
        -3.44081078e-07,  5.70669000e-06, -6.74147278e-12,
         4.88094410e-06,  2.37383227e-06,  1.15275632e-06,
        -1.96279855e-10,  3.35524901e-12,  6.57457622e-10,
         3.37960650e-10, -5.91030373e-10,  8.36161669e-07,
         1.07062320e-09,  2.55551081e-09, -1.43688378e-06,
         6.43789820e-09, -1.14650941e-14, -1.93403354e-10,
         3.50959877e-10, -4.90787809e-08,  1.98955754e-06,
         3.32211264e-10,  3.48205384e-10,  2.53778635e-10,
         4.69377067e-10, -2.89562451e-08,  7.76342317e-09,
         6.92152902e-11, -1.69228708e-12, -2.37921587e-10,
         7.58778939e-10,  3.36348958e-07,  7.83598950e-09,
         6.29212911e-11, -3.10727462e-12, -2.27379954e-10,
         4.89164982e-10,  4.119185

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.33681814022947887, 'recall': 0.7990355567274883, 'precision': 0.0909180305511848, 'specificity': 0.29610051142096694, 'f1_score': 0.16325278885569947, 'roc_auc': 0.5788456582922454}, None, {'accuracy': np.float64(0.33170381958023726), 'recall': np.float64(0.7834251888140437), 'precision': np.float64(0.08764758272625545), 'specificity': np.float64(0.29251447696966476), 'f1_score': np.float64(0.1576569175156096), 'roc_auc': 0.5709181854845268, 'threshold_used': 0.073063934296911})


/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integra

({'accuracy': 0.38376087684399807, 'recall': 0.7998188846232599, 'precision': 0.09740352466066313, 'specificity': 0.3471171628857624, 'f1_score': 0.17365116478125087, 'roc_auc': 0.6210451533800994}, array([[-6.11674010e-08, -5.71578153e-09, -1.98306422e-08,
        -3.59608513e-07,  5.65866678e-06, -1.02939975e-10,
         5.59630344e-05,  1.60399059e-05,  6.78287768e-05,
        -7.45474521e-09, -3.67053083e-10, -9.07535286e-09,
         1.77109398e-09, -8.60121197e-09,  2.11622895e-04,
         1.17455621e-09, -5.76475855e-08, -7.68695099e-07,
         1.29024986e-08, -3.40180290e-10, -1.06092407e-09,
        -1.43679727e-10,  1.20656215e-07,  1.50537217e-06,
        -3.44489108e-09, -3.30517307e-09, -1.30025048e-09,
        -2.75102948e-08, -7.88108513e-08, -1.34251989e-08,
         1.56334277e-09, -1.49431681e-10,  4.74467816e-10,
        -4.56027377e-08, -4.57644547e-08, -2.32004213e-10,
         1.47877388e-09, -2.42054134e-10, -2.38705525e-10,
        -4.05984676e-10, -3.614849

/tmp/ipython-input-2251938596.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))
